# Import libraries

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Data loading

In [2]:
GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings.csv не знайдено локально — завантажую з GitHub...
books.csv не знайдено локально — завантажую з GitHub...
book_tags.csv не знайдено локально — завантажую з GitHub...
tags.csv не знайдено локально — завантажую з GitHub...
ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


# Feature engineering from tags

In [3]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


# Subsample formation

In [4]:
TOP_BOOKS = 1500       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 140,934 | користувачів: 2,000 | книг: 1,496
Щільність: 0.0471


In [5]:
torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 77,070 | користувачів з val-лайками: 1,991


# Ranking quality assessment metric

In [6]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

# Vector Space Model

In [7]:
# L2-нормалізовані жанрові вектори книг
item_emb = item_feats / item_feats.norm(dim=1, keepdim=True).clamp(min=1e-8)
item_emb

tensor([[0.4082, 0.4082, 0.0000,  ..., 0.0000, 0.4082, 0.0000],
        [0.4472, 0.0000, 0.4472,  ..., 0.4472, 0.4472, 0.0000],
        [0.4472, 0.0000, 0.0000,  ..., 0.0000, 0.4472, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.7071, 0.0000, 0.0000],
        [0.5774, 0.0000, 0.0000,  ..., 0.5774, 0.0000, 0.0000],
        [0.4082, 0.0000, 0.4082,  ..., 0.0000, 0.4082, 0.4082]])

In [8]:
user_likes = {}
for row in train_pos.itertuples():
    u = user_to_idx[row.user_id]
    i = item_to_idx[row.book_id]
    user_likes.setdefault(u, []).append((i, float(row.rating)))

for u in list(user_likes.keys())[:5]:
    print(f"user_idx={u}: {user_likes[u]}")

user_idx=1812: [(256, 4.0), (494, 4.0), (939, 4.0), (119, 5.0), (101, 4.0), (856, 4.0), (95, 5.0), (196, 4.0), (136, 5.0), (71, 4.0), (241, 5.0), (299, 5.0), (82, 4.0), (524, 4.0), (294, 5.0), (388, 5.0), (809, 4.0), (764, 4.0), (7, 4.0), (62, 4.0), (15, 5.0), (571, 4.0), (395, 5.0), (427, 4.0), (499, 5.0), (1046, 5.0), (1361, 4.0), (642, 4.0), (192, 4.0), (8, 4.0), (581, 4.0), (210, 4.0), (102, 5.0), (158, 4.0), (41, 4.0), (255, 4.0), (1025, 4.0), (170, 5.0), (202, 4.0), (140, 4.0), (715, 4.0), (54, 4.0), (171, 5.0), (3, 5.0), (347, 4.0), (156, 5.0), (246, 4.0), (21, 5.0), (13, 4.0), (875, 4.0), (428, 4.0), (12, 4.0), (802, 4.0), (59, 5.0), (6, 5.0), (174, 5.0), (474, 4.0), (551, 5.0), (1210, 4.0), (175, 4.0), (362, 5.0), (437, 4.0), (114, 4.0), (333, 5.0), (28, 4.0), (407, 4.0), (53, 5.0), (1044, 5.0), (555, 5.0), (400, 4.0), (187, 5.0), (1081, 4.0), (178, 4.0), (61, 5.0), (4, 4.0), (344, 4.0), (291, 4.0), (217, 4.0), (129, 4.0), (542, 4.0), (638, 4.0), (584, 4.0), (573, 4.0), (355, 

In [9]:
def user_vector(user_idx):
    """Зважений (за оцінкою) середній ембединг уподобаних книг користувача."""
    likes = user_likes.get(user_idx, [])
    if not likes:
        return torch.zeros(n_genres)
    idxs = torch.tensor([i for i, _ in likes])
    weights = torch.tensor([w for _, w in likes])
    vecs = item_emb[idxs]                                  # (n_likes, n_genres)
    return (vecs * weights.unsqueeze(1)).sum(dim=0) / weights.sum()

def vsm_scores(user_idxs):
    """Косинусна подібність векторів користувачів до всіх книг. (B, M)"""
    if torch.is_tensor(user_idxs):
        user_idxs = user_idxs.tolist()
    user_vecs = torch.stack([user_vector(u) for u in user_idxs])      # (B, n_genres)
    user_vecs = user_vecs / user_vecs.norm(dim=1, keepdim=True).clamp(min=1e-8)
    return user_vecs @ item_emb.T                                     # (B, M)

In [10]:
def genres_of(item_idx):
    """Список жанрів книги за її item_idx (з item_feats)."""
    g = item_feats[item_idx]
    names = [GENRES[k] for k in range(len(GENRES)) if g[k] > 0]
    return ", ".join(names) if names else "—"

# Топ-5 рекомендацій
example_user = list(val_pos.keys())[0]
scores = vsm_scores(torch.tensor([example_user])).squeeze(0).clone()
for i in seen_by_user[example_user]:
    scores[i] = -1e9

top5 = torch.topk(scores, 5).indices.tolist()
rec_df = pd.DataFrame([
    {
        "rank": rank,
        "title": title_of.get(items[i], "???"),
        "score": round(scores[i].item(), 3),
        "genres": genres_of(i),
    }
    for rank, i in enumerate(top5, 1)
])

print(f"Користувач idx={example_user}")
print("\nРекомендовано (VSM):")
display(rec_df)

Користувач idx=316

Рекомендовано (VSM):


,rank,title,score,genres
0,1,The Battle of the Labyrinth (Percy Jackson and...,0.943,"fantasy, romance, mystery, young-adult, contem..."
1,2,"Hush, Hush (Hush, Hush, #1)",0.943,"fantasy, romance, mystery, young-adult, contem..."
2,3,"Fallen (Fallen, #1)",0.943,"fantasy, romance, mystery, young-adult, contem..."
3,4,"Shadow Kiss (Vampire Academy, #3)",0.943,"fantasy, romance, mystery, young-adult, contem..."
4,5,"Beautiful Creatures (Caster Chronicles, #1)",0.943,"fantasy, romance, mystery, young-adult, contem..."


In [11]:
# Топ-5 реально вподобаних книг цього користувача (за рейтингом) ---
liked_sorted = sorted(user_likes.get(example_user, []), key=lambda x: -x[1])[:5]
liked_df = pd.DataFrame([
    {
        "rank": rank,
        "title": title_of.get(items[i], "???"),
        "rating": rating,
        "genres": genres_of(i),
    }
    for rank, (i, rating) in enumerate(liked_sorted, 1)
])

print("\nРеально вподобано (train):")
display(liked_df)


Реально вподобано (train):


,rank,title,rating,genres
0,1,The Sea of Monsters (Percy Jackson and the Oly...,5.0,"fantasy, romance, young-adult, contemporary"
1,2,The Last Olympian (Percy Jackson and the Olymp...,5.0,"fantasy, romance, young-adult, contemporary"
2,3,Harry Potter and the Goblet of Fire (Harry Pot...,5.0,"fantasy, mystery, young-adult, classics, conte..."
3,4,Harry Potter and the Sorcerer's Stone (Harry P...,5.0,"fantasy, mystery, young-adult, classics, conte..."
4,5,Harry Potter and the Half-Blood Prince (Harry ...,5.0,"fantasy, romance, mystery, young-adult, classi..."


In [12]:
def genre_summary(user_idx):
    """Таблиця: для кожного жанру -- скільки книг цього жанру оцінив користувач
    і яка середня оцінка."""
    likes = user_likes.get(user_idx, [])
    rows = []
    for g_idx, g_name in enumerate(GENRES):
        ratings = [r for i, r in likes if item_feats[i, g_idx] > 0]
        rows.append({
            "genre": g_name,
            "count": len(ratings),
            "avg_rating": round(sum(ratings) / len(ratings), 2) if ratings else None,
        })
    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

print(f"Жанровий профіль користувача idx={example_user}:")
display(genre_summary(example_user))

Жанровий профіль користувача idx=316:


,genre,count,avg_rating
0,young-adult,22,4.55
1,fantasy,20,4.55
2,contemporary,17,4.59
3,romance,14,4.57
4,mystery,14,4.64
5,science-fiction,8,4.12
6,classics,7,4.71
7,historical,5,4.20
8,nonfiction,2,4.50
9,crime,1,4.00


In [13]:
# Recall@10
recall10_vsm = recall_at_k(vsm_scores, k=10)
print(f"VSM Recall@10: {recall10_vsm:.4f}")

VSM Recall@10: 0.0522


Recall@10 takes into account only those books that the user actually rated in the validation sample. However, the model may recommend a book that the user has not yet rated, although the rating may be positive, accordingly, this may be taken into account in the metric as an incorrect recommendation due to incomplete data, which lowers the quality of the model.

Currently, only 12 genres are used from the features to describe books. At the same time, certain books may have similar genre vectors, as a result of which they will be the same for the models. At the same time, the model may guess the correct direction of genres, but not a specific book among other similar ones. Accordingly, recall is intended to indicate the correctness of the book recommendation, while the model may actually issue a genre recommendation.

# Two-Tower architecture

In [14]:
EMB_DIM = 32
TEMPERATURE = 10.0

class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=EMB_DIM):
        super().__init__()
        self.user_tower = nn.Embedding(n_users, emb_dim)
        self.item_tower = nn.Sequential(
            nn.Linear(n_genres, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def user_vec(self, user_idx):
        v = self.user_tower(user_idx)
        return v / v.norm(dim=-1, keepdim=True).clamp(min=1e-8)

    def item_vec(self, feats):
        v = self.item_tower(feats)
        return v / v.norm(dim=-1, keepdim=True).clamp(min=1e-8)

    def forward(self, user_idx, item_idx):
        u = self.user_vec(user_idx)
        i = self.item_vec(item_feats[item_idx])
        return (u * i).sum(dim=-1) * TEMPERATURE

In [15]:
torch.manual_seed(42)
model_tt = TwoTower(n_users=len(users), n_genres=n_genres)
optimizer = torch.optim.Adam(model_tt.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

N_EPOCHS = 60
BATCH_SIZE = 512
n_pos = len(pos_u)
best_recall = -1.0
best_state = None

for epoch in range(N_EPOCHS):
    perm = torch.randperm(n_pos)
    total_loss = 0.0
    for start in range(0, n_pos, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        batch_u = pos_u[idx]
        batch_i_pos = pos_i[idx]
        batch_i_neg = torch.randint(0, M, (len(idx),))

        users_cat = torch.cat([batch_u, batch_u])
        items_cat = torch.cat([batch_i_pos, batch_i_neg])
        labels = torch.cat([torch.ones(len(idx)), torch.zeros(len(idx))])

        logits = model_tt(users_cat, items_cat)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)

    if (epoch + 1) % 5 == 0:
        model_tt.eval()
        with torch.no_grad():
            item_vecs_tmp = model_tt.item_vec(item_feats)
        recall_tmp = recall_at_k(
            lambda u: model_tt.user_vec(u) @ item_vecs_tmp.T, k=10
        )
        model_tt.train()

        if recall_tmp > best_recall:
            best_recall = recall_tmp
            best_state = {k: v.clone() for k, v in model_tt.state_dict().items()}

        print(f"epoch {epoch+1}: loss={total_loss/n_pos:.4f}  recall@10={recall_tmp:.4f}")

model_tt.load_state_dict(best_state)
model_tt.eval()
print(f"\nНайкращий recall@10={best_recall:.4f}")

epoch 5: loss=0.5823  recall@10=0.0509
epoch 10: loss=0.5415  recall@10=0.0578
epoch 15: loss=0.5218  recall@10=0.0658
epoch 20: loss=0.5092  recall@10=0.0693
epoch 25: loss=0.4983  recall@10=0.0695
epoch 30: loss=0.4894  recall@10=0.0684
epoch 35: loss=0.4823  recall@10=0.0743
epoch 40: loss=0.4766  recall@10=0.0705
epoch 45: loss=0.4683  recall@10=0.0723
epoch 50: loss=0.4665  recall@10=0.0687
epoch 55: loss=0.4619  recall@10=0.0720
epoch 60: loss=0.4575  recall@10=0.0682

Найкращий recall@10=0.0743


In [16]:
# Рекомендації
model_tt.eval()
with torch.no_grad():
    item_vecs_tt = model_tt.item_vec(item_feats)

def two_tower_scores(user_idxs):
    with torch.no_grad():
        user_vecs = model_tt.user_vec(user_idxs)
    return user_vecs @ item_vecs_tt.T

example_user = list(val_pos.keys())[0]
scores = two_tower_scores(torch.tensor([example_user])).squeeze(0).clone()
for i in seen_by_user[example_user]:
    scores[i] = -1e9

top5 = torch.topk(scores, 5).indices.tolist()
rec_df_tt = pd.DataFrame([
    {
        "rank": rank,
        "title": title_of.get(items[i], "???"),
        "score": round(scores[i].item(), 3),
        "genres": genres_of(i),
    }
    for rank, i in enumerate(top5, 1)
])
print(f"\nTwo-Tower рекомендації для користувача idx={example_user}")
display(rec_df_tt)


Two-Tower рекомендації для користувача idx=316


,rank,title,score,genres
0,1,"Inkheart (Inkworld, #1)",0.378,"fantasy, romance, mystery, young-adult, classi..."
1,2,"Heir of Fire (Throne of Glass, #3)",0.372,"fantasy, romance, mystery, young-adult"
2,3,"Throne of Glass (Throne of Glass, #1)",0.372,"fantasy, romance, mystery, young-adult"
3,4,"Crown of Midnight (Throne of Glass, #2)",0.372,"fantasy, romance, mystery, young-adult"
4,5,Harry Potter and the Order of the Phoenix (Har...,0.372,"fantasy, romance, mystery, young-adult"


In [17]:
print("\nРеально вподобано (train):")
display(liked_df)

print(f"Жанровий профіль користувача idx={example_user}:")
display(genre_summary(example_user))


Реально вподобано (train):


,rank,title,rating,genres
0,1,The Sea of Monsters (Percy Jackson and the Oly...,5.0,"fantasy, romance, young-adult, contemporary"
1,2,The Last Olympian (Percy Jackson and the Olymp...,5.0,"fantasy, romance, young-adult, contemporary"
2,3,Harry Potter and the Goblet of Fire (Harry Pot...,5.0,"fantasy, mystery, young-adult, classics, conte..."
3,4,Harry Potter and the Sorcerer's Stone (Harry P...,5.0,"fantasy, mystery, young-adult, classics, conte..."
4,5,Harry Potter and the Half-Blood Prince (Harry ...,5.0,"fantasy, romance, mystery, young-adult, classi..."


Жанровий профіль користувача idx=316:


,genre,count,avg_rating
0,young-adult,22,4.55
1,fantasy,20,4.55
2,contemporary,17,4.59
3,romance,14,4.57
4,mystery,14,4.64
5,science-fiction,8,4.12
6,classics,7,4.71
7,historical,5,4.20
8,nonfiction,2,4.50
9,crime,1,4.00


# Concat-based ranking (NCF)

In [18]:
class NCF(nn.Module):
    def __init__(self, n_users, n_genres, emb_dim=32, hidden=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_genres, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, user_idx, item_idx):
        u = self.user_emb(user_idx)          # (B, emb_dim)
        i = item_feats[item_idx]             # (B, n_genres)
        x = torch.cat([u, i], dim=-1)        # (B, emb_dim + n_genres) — early fusion
        return self.mlp(x).squeeze(-1)       # (B,) — логіт

In [19]:
def ncf_scores(user_idxs):
    """Оцінки NCF (sigmoid-логіт) для всіх M книг для пачки користувачів. (B, M)
    Дорожче, ніж у Two-Tower, бо немає окремих векторів — кожну пару (user, item) рахуємо явно."""
    if torch.is_tensor(user_idxs):
        user_idxs = user_idxs.tolist()
    B = len(user_idxs)
    user_t = torch.tensor(user_idxs).repeat_interleave(M)  # (B*M,)
    item_t = torch.arange(M).repeat(B)                     # (B*M,)
    with torch.no_grad():
        logits = model_ncf(user_t, item_t)
    return torch.sigmoid(logits).view(B, M)

In [20]:
torch.manual_seed(42)
model_ncf = NCF(n_users=len(users), n_genres=n_genres)
optimizer = torch.optim.Adam(model_ncf.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()

N_EPOCHS = 100
BATCH_SIZE = 512
n_pos = len(pos_u)
best_recall = -1.0
best_state = None

for epoch in range(N_EPOCHS):
    perm = torch.randperm(n_pos)
    total_loss = 0.0
    for start in range(0, n_pos, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        batch_u = pos_u[idx]
        batch_i_pos = pos_i[idx]
        batch_i_neg = torch.randint(0, M, (len(idx),))

        users_cat = torch.cat([batch_u, batch_u])
        items_cat = torch.cat([batch_i_pos, batch_i_neg])
        labels = torch.cat([torch.ones(len(idx)), torch.zeros(len(idx))])

        logits = model_ncf(users_cat, items_cat)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)

    if (epoch + 1) % 10 == 0:
        model_ncf.eval()
        recall_tmp = recall_at_k(ncf_scores, k=10)
        model_ncf.train()

        if recall_tmp > best_recall:
            best_recall = recall_tmp
            best_state = {k: v.clone() for k, v in model_ncf.state_dict().items()}

        print(f"epoch {epoch+1}: loss={total_loss/n_pos:.4f}  recall@10={recall_tmp:.4f}")

model_ncf.load_state_dict(best_state)
model_ncf.eval()
print(f"\nНайкращий recall@10={best_recall:.4f}")

epoch 10: loss=0.6012  recall@10=0.0415
epoch 20: loss=0.5608  recall@10=0.0469
epoch 30: loss=0.5438  recall@10=0.0500
epoch 40: loss=0.5311  recall@10=0.0514
epoch 50: loss=0.5240  recall@10=0.0533
epoch 60: loss=0.5164  recall@10=0.0529
epoch 70: loss=0.5108  recall@10=0.0536
epoch 80: loss=0.5058  recall@10=0.0543
epoch 90: loss=0.5031  recall@10=0.0532
epoch 100: loss=0.4998  recall@10=0.0551

Найкращий recall@10=0.0551


In [21]:
# rank_ncf(user_idx, candidate_idxs)
def rank_ncf(user_idx, candidate_idxs):
    """Ранжує задані candidate_idxs за sigmoid(логіт) NCF. Повертає список (item_idx, prob), за спаданням."""
    with torch.no_grad():
        user_t = torch.full((len(candidate_idxs),), user_idx, dtype=torch.long)
        cand_t = torch.tensor(candidate_idxs, dtype=torch.long)
        probs = torch.sigmoid(model_ncf(user_t, cand_t))
    order = torch.argsort(probs, descending=True).tolist()
    return [(candidate_idxs[i], probs[i].item()) for i in order]

In [22]:
example_user = list(val_pos.keys())[0]
candidates = [i for i in range(M) if i not in seen_by_user[example_user]]

ranked = rank_ncf(example_user, candidates)[:5]
rank_df_ncf = pd.DataFrame([
    {
        "rank": r + 1,
        "title": title_of.get(items[i], "???"),
        "prob": round(p, 3),
        "genres": genres_of(i),
    }
    for r, (i, p) in enumerate(ranked)
])
print(f"NCF — топ-5 ранжування для користувача idx={example_user}")
display(rank_df_ncf)

NCF — топ-5 ранжування для користувача idx=316


,rank,title,prob,genres
0,1,"The One (The Selection, #3)",0.971,"fantasy, romance, science-fiction, young-adult..."
1,2,"Scott Pilgrim, Volume 1: Scott Pilgrim's Preci...",0.971,"fantasy, romance, science-fiction, young-adult..."
2,3,"Obsidian (Lux, #1)",0.971,"fantasy, romance, science-fiction, young-adult..."
3,4,Landline,0.971,"fantasy, romance, science-fiction, young-adult..."
4,5,"Opal (Lux, #3)",0.971,"fantasy, romance, science-fiction, young-adult..."


In [23]:
print("\nРеально вподобано (train):")
display(liked_df)

print(f"Жанровий профіль користувача idx={example_user}:")
display(genre_summary(example_user))


Реально вподобано (train):


,rank,title,rating,genres
0,1,The Sea of Monsters (Percy Jackson and the Oly...,5.0,"fantasy, romance, young-adult, contemporary"
1,2,The Last Olympian (Percy Jackson and the Olymp...,5.0,"fantasy, romance, young-adult, contemporary"
2,3,Harry Potter and the Goblet of Fire (Harry Pot...,5.0,"fantasy, mystery, young-adult, classics, conte..."
3,4,Harry Potter and the Sorcerer's Stone (Harry P...,5.0,"fantasy, mystery, young-adult, classics, conte..."
4,5,Harry Potter and the Half-Blood Prince (Harry ...,5.0,"fantasy, romance, mystery, young-adult, classi..."


Жанровий профіль користувача idx=316:


,genre,count,avg_rating
0,young-adult,22,4.55
1,fantasy,20,4.55
2,contemporary,17,4.59
3,romance,14,4.57
4,mystery,14,4.64
5,science-fiction,8,4.12
6,classics,7,4.71
7,historical,5,4.20
8,nonfiction,2,4.50
9,crime,1,4.00


# Two-stage pipeline Retrieval → Ranking

In [24]:
def retrieve(user_idx, n_candidates=50):
    """Топ-N книг за Two-Tower (косинусна подібність), без уже побачених."""
    with torch.no_grad():
        user_vec = model_tt.user_vec(torch.tensor([user_idx]))
        scores = (user_vec @ item_vecs_tt.T).squeeze(0).clone()
    for i in seen_by_user[user_idx]:
        scores[i] = -1e9
    top_idxs = torch.topk(scores, n_candidates).indices.tolist()
    return top_idxs


def recommend_pipeline(user_idx, n_candidates=50, top_k=5):
    """Retrieval (Two-Tower) -> Ranking (NCF). Повертає top_k (item_idx, prob)."""
    candidates = retrieve(user_idx, n_candidates)
    ranked = rank_ncf(user_idx, candidates)
    return ranked[:top_k]

In [25]:
example_users = list(val_pos.keys())[:3]
N_CANDIDATES = 50
TOP_K = 5

for u in example_users:
    candidates = retrieve(u, N_CANDIDATES)
    final = recommend_pipeline(u, N_CANDIDATES, TOP_K)
    final_idxs = {i for i, _ in final}

    rows = []
    for rank, i in enumerate(candidates, 1):
        rows.append({
            "retrieval_rank": rank,
            "title": title_of.get(items[i], "???"),
            "genres": genres_of(i),
            "у фінальному топ-5 (NCF)": "так" if i in final_idxs else "ні",
        })
    df = pd.DataFrame(rows)

    print(f"\nКористувач idx={u}")
    print(f"Retrieval (Two-Tower) відібрав {N_CANDIDATES} кандидатів, NCF переранжував і лишив топ-{TOP_K}:")
    display(df)

    final_df = pd.DataFrame([
        {"rank": r + 1, "title": title_of.get(items[i], "???"), "prob": round(p, 3), "genres": genres_of(i)}
        for r, (i, p) in enumerate(final)
    ])
    print(f"Фінальний топ-{TOP_K} після ranking:")
    display(final_df)

    # реальні вподобання користувача (топ-5 за рейтингом, train)
    liked_sorted = sorted(user_likes.get(u, []), key=lambda x: -x[1])[:TOP_K]
    liked_df = pd.DataFrame([
        {"rank": r + 1, "title": title_of.get(items[i], "???"), "rating": rating, "genres": genres_of(i)}
        for r, (i, rating) in enumerate(liked_sorted)
    ])
    print(f"Реально вподобані книги (train, топ-{TOP_K} за рейтингом):")
    display(liked_df)

    # повний жанровий профіль цього користувача (усі лайки, не лише топ-5)
    print(f"Жанровий профіль користувача idx={u} (увесь train):")
    display(genre_summary(u))


Користувач idx=316
Retrieval (Two-Tower) відібрав 50 кандидатів, NCF переранжував і лишив топ-5:


,retrieval_rank,title,genres,у фінальному топ-5 (NCF)
0,1,"Inkheart (Inkworld, #1)","fantasy, romance, mystery, young-adult, classi...",ні
1,2,"Heir of Fire (Throne of Glass, #3)","fantasy, romance, mystery, young-adult",ні
2,3,"Throne of Glass (Throne of Glass, #1)","fantasy, romance, mystery, young-adult",ні
3,4,"Bitterblue (Graceling Realm, #3)","fantasy, romance, mystery, young-adult",ні
4,5,"Crown of Midnight (Throne of Glass, #2)","fantasy, romance, mystery, young-adult",ні
5,6,Harry Potter and the Order of the Phoenix (Har...,"fantasy, romance, mystery, young-adult",ні
6,7,"A Great and Terrible Beauty (Gemma Doyle, #1)","fantasy, romance, mystery, historical, young-a...",ні
7,8,Harry Potter and the Cursed Child - Parts One ...,"fantasy, mystery, young-adult, contemporary",ні
8,9,"The Book of Life (All Souls Trilogy, #3)","fantasy, romance, mystery, historical, science...",ні
9,10,"The Golem's Eye (Bartimaeus, #2)","fantasy, mystery, young-adult",ні


Фінальний топ-5 після ranking:


,rank,title,prob,genres
0,1,"Every Day (Every Day, #1)",0.971,"fantasy, romance, science-fiction, young-adult..."
1,2,"The Heir (The Selection, #4)",0.971,"fantasy, romance, science-fiction, young-adult..."
2,3,"Midnight Sun (Twilight, #1.5)",0.971,"fantasy, romance, science-fiction, young-adult..."
3,4,"The One (The Selection, #3)",0.971,"fantasy, romance, science-fiction, young-adult..."
4,5,"The Elite (The Selection, #2)",0.971,"fantasy, romance, science-fiction, young-adult..."


Реально вподобані книги (train, топ-5 за рейтингом):


...

,rank,title,prob,genres
0,1,"Temple of the Winds (Sword of Truth, #4)",0.855,"fantasy, romance, science-fiction"
1,2,"Lord of Chaos (Wheel of Time, #6)",0.855,"fantasy, romance, science-fiction"
2,3,"Words of Radiance (The Stormlight Archive, #2)",0.855,"fantasy, romance, science-fiction"
3,4,"Stone of Tears (Sword of Truth, #2)",0.855,"fantasy, romance, science-fiction"
4,5,"Blood of the Fold (Sword of Truth, #3)",0.855,"fantasy, romance, science-fiction"


Реально вподобані книги (train, топ-5 за рейтингом):


,rank,title,rating,genres
0,1,"A Dance with Dragons (A Song of Ice and Fire, #5)",5.0,"fantasy, science-fiction, contemporary"
1,2,"A Memory of Light (Wheel of Time, #14)",5.0,"fantasy, romance, science-fiction"
2,3,"The Gathering Storm (Wheel of Time, #12)",5.0,"fantasy, science-fiction"
3,4,"The Shadow Rising (Wheel of Time, #4)",5.0,"fantasy, science-fiction"
4,5,"Towers of Midnight (Wheel of Time, #13)",5.0,"fantasy, romance, science-fiction"


Жанровий профіль користувача idx=788 (увесь train):


,genre,count,avg_rating
0,fantasy,33,4.55
1,science-fiction,31,4.55
2,romance,17,4.59
3,young-adult,13,4.38
4,thriller,9,4.33
5,contemporary,8,4.25
6,classics,8,4.38
7,mystery,6,4.50
8,historical,5,4.40
9,nonfiction,4,4.50


NCF is a more resource-intensive approach when applied independently on the entire database, since to evaluate a book you need to run each "user - book" pair through MLP. While Two-Tower allows you to calculate the vectors of books at a time and put them in the index, as a result of which the search for TOP candidates becomes faster. So the two-stage scheme combines retrieval speed with ranking accuracy, taking into account the peculiarities of both aspects.

**Observation:**

According to the simulation results for user 316, we can see that the Recall@10 quality metric and the closeness of the recommendations to the real preferences of this user are modeling using Two-Tower. In this case, in the case of the two-stage pipeline, NCF highlighted books mainly in the romance and science-fiction genres, which are not the top genres of this user, compared to young-adult and fantasy, although they are present in his profile.

The Vector Space Model selected books according to the same genre vector without diversity.

However, to select a model, it would be worth analyzing several typical and atypical users to avoid bias on only one user. It also makes sense to try to enrich the data with new features and see if it is possible to improve the quality of recommendations.

**1. Why is Recall@10 so low?**
- Currently, only 12 genres are used to describe books. At the same time, certain books may have similar genre vectors, as a result of which they will be the same for the models. At the same time, the model may guess the correct direction of genres, but not a specific book among other similar ones. Accordingly, recall is intended to indicate the correctness of the book recommendation, while the model may actually issue a genre recommendation.
- A random negative that is close to the user's preferences can sometimes mislead the model.
- Recall@10 takes into account only those books that the user actually rated in the validation sample. However, the model may recommend a book that the user has not yet rated, although the rating may be positive, respectively, this may be taken into account in the metric as an incorrect recommendation due to incomplete data, which lowers the quality of the model.
- A certain part of users liked a small number of books.

**2. How to improve quality without changing the architecture?**
You can add:
- book metadata such as author, year of publication, language, number of pages
- aggregated features -- average rating of the book, number of ratings, share of 5-star ratings
- semantic embeddings -- description of books via BERT / Sentence-Transformers
- user features -- average rating of a specific user, his activity

**3. If a user likes fantasy, why not show him 10 fantasy books in a row?**

The user may not be interested in seeing only similar books in the recommendations, there will be a need for genre diversity, which the recommendation system will not be able to satisfy, as a result of which the user may interact less with our platform.

**How ​​to technically mix diversity?**
A limit on how many books of one genre can get into TOP-N. Set certain weighting factors for fairly similar books while forming the rating by the highest score.

**4. The new book has 0 ratings. Which approach can be recommended to her immediately, and which one cannot? Why?**
Any collaborative filtering approach may not work, since in its case we need a vector of user interaction, and in the absence of ratings the model will simply guess.
For practical implementation, Vector Space Model and Two-Tower may be suitable, which build a vector based solely on content features regardless of interaction history. NFC can also recommend a new book based on book characteristics, but it is not purely content-based.